# 🌊 Culvert & Slope Hydrological Precondition Engine
### Physics-Informed Transportation Infrastructure Resilience Under Extreme Precipitation
**Author:** Peter Aryee  
**Domain:** Computational Hydrology & Geotechnical Data Science  
**Target Research:** USDA Forest Service CulvertApp & Physics-Informed Landslide Risk

---

## 1. Executive Summary & Problem Formulation
Across mountain road networks, stream crossings rely on **culverts** to convey stream discharge beneath road embankments. During extreme precipitation events, culverts frequently fail via two coupled mechanisms:
1. **Hydraulic Surcharge:** Storm peak discharge $Q_{\text{peak}}$ exceeds pipe conveyance capacity $Q_{\text{cap}}$, causing water to pond behind the road embankment.
2. **Geotechnical Embankment / Slope Instability:** High antecedent soil moisture $m = z_w / z$ creates elevated pore-water pressures $u$, reducing effective normal stress ($\sigma' = \sigma - u$) and triggering landslides that liquefy or choke the culvert entrance.

This notebook demonstrates the end-to-end open-source pipeline linking **DEM terrain analysis**, **D8 hydrological routing**, **spatial vector road intersection**, and **Terzaghi-Coulomb slope stability**.


In [ ]:
import sys
from pathlib import Path
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

# Add src to system path
sys.path.append(str(Path.cwd().parent))

from src.hydro_routing import (
    fill_depressions,
    calculate_slope_and_aspect,
    calculate_d8_flow_direction,
    calculate_flow_accumulation,
    extract_stream_network
)
from src.road_intersection import (
    detect_culvert_crossings,
    extract_culvert_catchment_attributes
)
from src.slope_physics import (
    calculate_factor_of_safety,
    evaluate_moisture_sensitivity_curve
)
from src.hydraulic_capacity import (
    calculate_peak_storm_runoff,
    calculate_culvert_manning_capacity
)
from src.risk_engine import assess_culvert_network_risk
from src.visualizer import build_interactive_risk_map

print("✅ Successfully loaded all pipeline modules.")


## 2. Ingesting Digital Elevation Model & Computing Terrain Slopes
We ingest the georeferenced GeoTIFF raster and compute topographic slope gradient $\beta$ using centered finite differences.


In [ ]:
dem_path = Path("../data/sample_dem.tif")
with rasterio.open(dem_path) as src:
    dem = src.read(1)
    transform = src.transform
    crs = src.crs
    resolution = float(abs(src.res[0]))

print(f"DEM Grid: {dem.shape[1]}x{dem.shape[0]} at {resolution}m resolution")
print(f"Elevation Range: {dem.min():.1f}m to {dem.max():.1f}m")

# Hydro-condition and calculate slope
dem_filled = fill_depressions(dem)
slope_deg, slope_rad = calculate_slope_and_aspect(dem_filled, resolution)

# Plot Elevation & Slope
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
im1 = ax1.imshow(dem_filled, cmap='terrain')
ax1.set_title("A. Conditioned Digital Elevation Model (m)", fontweight='bold')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(slope_deg, cmap='magma', vmin=0, vmax=45)
ax2.set_title("B. Topographic Slope Gradient (Degrees)", fontweight='bold')
plt.colorbar(im2, ax=ax2)
plt.tight_layout()
plt.show()


## 3. D8 Flow Accumulation & Stream Network Extraction
Using steepest descent D8 routing, we calculate the contributing upstream drainage area for every cell and vectorize the continuous stream channel network.


In [ ]:
flow_dir = calculate_d8_flow_direction(dem_filled, resolution)
accum_cells, accum_km2 = calculate_flow_accumulation(dem_filled, flow_dir, resolution)
streams_gdf = extract_stream_network(accum_km2, transform, crs.to_string(), threshold_km2=0.35)

print(f"Extracted {len(streams_gdf)} stream reach segments.")

# Plot Flow Accumulation and Extracted Streams
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(np.log10(accum_cells + 1), cmap='Blues')
streams_gdf.plot(ax=ax, color='red', linewidth=1.5, label='Stream Channels (>= 0.35 km²)')
ax.set_title("D8 Log Flow Accumulation & Stream Network", fontweight='bold')
plt.colorbar(im, ax=ax, label='Log10(Contributing Cells)')
plt.legend()
plt.show()


## 4. Road Vector Intersections & Culvert Candidate Detection
We load the road vector network (LineStrings) and perform a spatial intersection with the stream network to locate and attribute all culverts.


In [ ]:
roads_path = Path("../data/sample_roads.geojson")
roads_gdf = gpd.read_file(roads_path)

crossings_gdf = detect_culvert_crossings(roads_gdf, streams_gdf)
print(f"Identified {len(crossings_gdf)} culvert crossings.")

attributed_culverts = extract_culvert_catchment_attributes(
    crossings_gdf, dem_filled, slope_deg, flow_dir, accum_km2, transform, resolution
)

attributed_culverts[['culvert_id', 'road_name', 'drainage_area_km2', 'p90_upslope_deg']].head(6)


## 5. Terzaghi-Coulomb Slope Factor of Safety Under Antecedent Moisture
We model the geotechnical Factor of Safety ($FS$) as a function of the antecedent soil saturation ratio $m = z_w / z$:

$$FS = \frac{c' + (\gamma_s - m \gamma_w) z \cos^2\beta \tan\phi'}{\gamma_s z \sin\beta \cos\beta}$$


In [ ]:
# Evaluate sensitivity across moisture states for a steep slope (38 deg) vs moderate slope (20 deg)
m_vals = np.linspace(0.0, 1.0, 50)
fs_steep = [calculate_factor_of_safety(38.0, saturation_ratio_m=m) for m in m_vals]
fs_moderate = [calculate_factor_of_safety(20.0, saturation_ratio_m=m) for m in m_vals]

plt.figure(figsize=(9, 5))
plt.plot(m_vals, fs_steep, 'r-', lw=2.5, label='Steep Embankment / Hollow (38° slope)')
plt.plot(m_vals, fs_moderate, 'b-', lw=2.5, label='Moderate Mountain Slope (20° slope)')
plt.axhline(1.0, color='red', linestyle='--', lw=2, label='Failure Limit (FS = 1.0)')
plt.axhline(1.5, color='green', linestyle=':', lw=1.5, label='Civil Design Target (FS = 1.5)')
plt.title("Geotechnical Stability Degradation Under Hydrologic Saturation", fontweight='bold')
plt.xlabel("Antecedent Saturation Ratio (m = zw / z)")
plt.ylabel("Factor of Safety (FS)")
plt.ylim(0.2, 3.5)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 6. Multi-Hazard Culvert Risk Scoring & Interactive Web GIS Map
We run the integrated assessment evaluating both geotechnical slope instability and hydraulic surcharge under a 75 mm/hr design storm with 70% antecedent moisture.


In [ ]:
assessed_culverts = assess_culvert_network_risk(
    attributed_culverts,
    antecedent_saturation_m=0.70,
    design_rainfall_intensity_mm_hr=75.0,
    default_pipe_diameter_m=1.2
)

# Display Top Vulnerable Culverts
assessed_culverts[[
    'culvert_id', 'road_name', 'p90_upslope_deg', 'factor_of_safety',
    'q_peak_m3s', 'hydraulic_surcharge_ratio', 'composite_vulnerability_index', 'risk_tier'
]].head(10)


## 7. Interactive Folium Web Map Preview
We export an interactive web GIS map with color-coded risk pins and popup engineering cards.


In [ ]:
html_out = Path("../output/culvert_risk_map.html")
build_interactive_risk_map(assessed_culverts, roads_gdf, streams_gdf, html_out)
print(f"✅ Generated interactive web map at: {html_out}")
